In [2]:
import os
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT_DIR = os.path.abspath('')   # project root (where this notebook lives)
EXP_DIR  = os.path.join(ROOT_DIR, 'experiments',
                         'geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn')

# ── USER CONFIG ───────────────────────────────────────────────────────────────
PLY_PATH    = os.path.join(ROOT_DIR, 'UHM_generated_data', '0.ply')
OUTPUT_PATH = os.path.join(EXP_DIR, '0_face.npy')
# ─────────────────────────────────────────────────────────────────────────────

print('PLY   :', PLY_PATH)
print('Output:', OUTPUT_PATH)

PLY   : c:\Eli Folder temp\geotransformer-faces-updated\UHM_generated_data\0.ply
Output: c:\Eli Folder temp\geotransformer-faces-updated\experiments\geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn\0_face.npy


In [3]:
# Load full head
pcd = o3d.io.read_point_cloud(PLY_PATH)
pts = np.asarray(pcd.points).astype(np.float32)
print(f'Points loaded : {pts.shape[0]}')
print()
print('Axis ranges (inspect these to set crop bounds below):')
for i, ax in enumerate(['X', 'Y', 'Z']):
    print(f'  {ax}  [{pts[:, i].min():.4f},  {pts[:, i].max():.4f}]')

Points loaded : 71926

Axis ranges (inspect these to set crop bounds below):
  X  [-0.9382,  0.9607]
  Y  [-1.3095,  1.1818]
  Z  [-1.3188,  0.6413]


In [4]:
# ── Visualize full head from three viewpoints ─────────────────────────────────
def pcd_trace(pts, color, name, size=1.5, opacity=0.6):
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity),
        name=name,
    )

def show_cloud(traces, title, camera=None):
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title, height=550,
        scene=dict(aspectmode='data', camera=camera or {}),
        legend=dict(itemsizing='constant'),
        margin=dict(l=0, r=0, b=0, t=40),
    )
    fig.show()

# Subsample for speed if the mesh is dense
MAX_VIZ = 30_000
idx_viz = np.random.choice(len(pts), min(MAX_VIZ, len(pts)), replace=False)
pts_viz = pts[idx_viz]

# Front view (looking down -Z)
show_cloud([pcd_trace(pts_viz, 'steelblue', 'full head')],
           'Full head — front view',
           camera=dict(eye=dict(x=0, y=0, z=2)))

# Side view (looking down -X)
show_cloud([pcd_trace(pts_viz, 'steelblue', 'full head')],
           'Full head — side view (use this to pick Z crop threshold)',
           camera=dict(eye=dict(x=2, y=0, z=0)))

# Top view (looking down -Y)
show_cloud([pcd_trace(pts_viz, 'steelblue', 'full head')],
           'Full head — top view (use this to pick Y crop threshold)',
           camera=dict(eye=dict(x=0, y=2, z=0)))

In [62]:
import numpy as np
# ── USER CONFIG: axis-aligned crop bounds + subsampling ───────────────────────
# Set a bound to -np.inf / np.inf to leave that side open.
# Look at the axis ranges printed above and the side/top views to choose values.
rng = np.random.default_rng() 
# Base bounds
X_MIN_BASE, X_MAX_BASE = -0.8, 0.8 #-0.5,  0.5
Y_MIN_BASE, Y_MAX_BASE = -0.75, 1.1   #-0.5,  0.5
Z_MIN_BASE             = -0.25            #0.2

# Randomize each finite bound

X_MIN = X_MIN_BASE * rng.uniform(0.5, 1.0)
X_MAX = X_MAX_BASE * rng.uniform(0.5, 1.0)
Y_MIN = Y_MIN_BASE * rng.uniform(0.5, 1.0)
Y_MAX = Y_MAX_BASE * rng.uniform(0.5, 1.0)
Z_MIN = Z_MIN_BASE * rng.uniform(0.5, 1.0)
Z_MAX= np.inf

SUBSAMPLE_FRAC = 0.25       # fraction of face points to keep (1.0 = keep all)
# ─────────────────────────────────────────────────────────────────────────────

mask = (
    (pts[:, 0] >= X_MIN) & (pts[:, 0] <= X_MAX) &
    (pts[:, 1] >= Y_MIN) & (pts[:, 1] <= Y_MAX) &
    (pts[:, 2] >= Z_MIN) & (pts[:, 2] <= Z_MAX)
)
face_pts_full = pts[mask]
rest_pts      = pts[~mask]

# Random subsampling
n_keep   = max(1, int(len(face_pts_full) * SUBSAMPLE_FRAC))
keep_idx = np.random.choice(len(face_pts_full), n_keep, replace=False)
face_pts = face_pts_full[keep_idx]

print(f'After crop       : {face_pts_full.shape[0]}  ({100*mask.mean():.1f}%)')
print(f'After subsample  : {face_pts.shape[0]}  ({100*SUBSAMPLE_FRAC:.0f}% of cropped)')
print(f'Removed by crop  : {rest_pts.shape[0]}')

After crop       : 45372  (63.1%)
After subsample  : 11343  (25% of cropped)
Removed by crop  : 26554


In [63]:
# ── Preview: kept (orange) vs removed (grey) ─────────────────────────────────
MAX_VIZ_EACH = 15_000

def subsample(arr, n):
    if len(arr) == 0:
        return arr
    idx = np.random.choice(len(arr), min(n, len(arr)), replace=False)
    return arr[idx]

traces = [pcd_trace(subsample(face_pts, MAX_VIZ_EACH), 'orange',    'face (kept)',    size=2)]
if len(rest_pts) > 0:
    traces.append(pcd_trace(subsample(rest_pts, MAX_VIZ_EACH), 'lightgrey', 'removed', size=1, opacity=0.3))

show_cloud(traces, f'Crop preview — {face_pts.shape[0]} pts kept')

In [64]:
# ── Save ──────────────────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

if OUTPUT_PATH.endswith('.npy'):
    np.save(OUTPUT_PATH, face_pts)
    print(f'Saved {face_pts.shape} → {OUTPUT_PATH}')
elif OUTPUT_PATH.endswith('.ply'):
    out_pcd = o3d.geometry.PointCloud()
    out_pcd.points = o3d.utility.Vector3dVector(face_pts)
    o3d.io.write_point_cloud(OUTPUT_PATH, out_pcd)
    print(f'Saved {face_pts.shape[0]} pts → {OUTPUT_PATH}')
else:
    raise ValueError(f'OUTPUT_PATH must end with .npy or .ply, got: {OUTPUT_PATH}')

Saved (11343, 3) → c:\Eli Folder temp\geotransformer-faces-updated\experiments\geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn\0_face.npy
